<a href="https://colab.research.google.com/github/jasonenkristo123/AIML-Learn/blob/main/SapBERT_Liu_Colab_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1 — Mount Drive, tentukan SATU path runtime yang konsisten
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, shutil, getpass, tempfile, re, json, time

REPO = Path('/content/reproduce_liu')
DRIVE_REPO = Path('/content/drive/MyDrive/reproduce_liu')  # folder yang terlihat pada screenshot
STORE = Path('/content/drive/MyDrive/sapbert_liu')         # penyimpanan persisten
GITHUB_REPO = 'jasonenkristo123/reproduce_liu'
SOURCE_MODE = 'github'  # 'github' untuk clone/pull; 'drive' jika GitHub belum bisa diakses

for folder in ('models', 'results', 'runs', 'data'):
    (STORE / folder).mkdir(parents=True, exist_ok=True)
print('Repo runtime:', REPO)
print('Drive storage:', STORE)
print('Source mode:', SOURCE_MODE)

Mounted at /content/drive
Repo runtime: /content/reproduce_liu
Drive storage: /content/drive/MyDrive/sapbert_liu
Source mode: github


In [ ]:
# 2 — Ambil kode ke /content; jangan jalankan Git di folder yang salah.
# GitHub private: gunakan token rahasia sementara, BUKAN https://TOKEN@github.com/...

def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

if SOURCE_MODE == 'github':
    url = f'https://github.com/{GITHUB_REPO}.git'
    token = get_secret('GITHUB_TOKEN')
    if not token:
        token = getpass.getpass('GitHub PAT (tekan Enter jika public): ').strip()

    with tempfile.TemporaryDirectory() as td:
        askpass = Path(td) / 'askpass.sh'
        askpass.write_text('''#!/bin/sh
case "$1" in
  *Username*) printf '%s\n' 'x-access-token' ;;
  *Password*) printf '%s\n' "$GH_TOKEN" ;;
esac
''')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GH_TOKEN': token or ''})
        if (REPO / '.git').is_dir():
            # Bersihkan kemungkinan remote lama yang mengandung PAT
            subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', url], check=True)
            status = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip()
            if status == ' M src/model_wrapper.py':
                # Reset hanya patch batch yang dibuat notebook ini, setelah diperiksa identik.
                original = subprocess.check_output(['git', '-C', str(REPO), 'show', 'HEAD:src/model_wrapper.py'], text=True)
                patch_pattern = r'(def embed_dense\([^\n]*?batch_size=)(\d+)'
                expected = re.sub(patch_pattern, lambda m: m.group(1) + str(min(64, int(m.group(2)))), original, count=1)
                if (REPO / 'src/model_wrapper.py').read_text(encoding='utf-8') == expected:
                    subprocess.run(['git', '-C', str(REPO), 'restore', '--', 'src/model_wrapper.py'], check=True)
                    status = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip()
            if status:
                raise RuntimeError('Ada perubahan lokal selain patch notebook. Commit / backup dulu sebelum pull.')
            subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True, env=env)
        elif REPO.exists():
            raise RuntimeError(f'{REPO} sudah ada tapi bukan Git repository. Pindahkan folder itu lebih dulu.')
        else:
            subprocess.run(['git', 'clone', '--depth', '1', url, str(REPO)], check=True, env=env)
    token = None
    del env
elif SOURCE_MODE == 'drive':
    if not (DRIVE_REPO / 'src' / 'model_wrapper.py').is_file():
        raise FileNotFoundError('Folder Drive tidak memiliki src/model_wrapper.py')
    if REPO.exists():
        raise RuntimeError('Folder /content/reproduce_liu sudah ada. Restart runtime untuk menyalin ulang dari Drive.')
    shutil.copytree(DRIVE_REPO, REPO, ignore=shutil.ignore_patterns('sapbert_model', 'wandb', 'runs', 'results'))
    # Jangan biarkan remote asal memuat token jika .git ikut tersalin.
    if (REPO / '.git').is_dir():
        subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', f'https://github.com/{GITHUB_REPO}.git'], check=False)
else:
    raise ValueError('SOURCE_MODE harus github atau drive')

for required in ('src/model_wrapper.py', 'evaluation/eval.py', 'evaluation/utils.py'):
    assert (REPO / required).is_file(), f'Tidak ada {required}'
print('Kode siap:', REPO)
if (REPO / '.git').is_dir():
    print('Git commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())

Kode siap: /content/reproduce_liu
Git commit: bbfc86c


In [ ]:

# 3 — Setup GPU dan dependency SapBERT
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan T4 GPU di Colab!")

print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

# Hapus library yang tidak digunakan SapBERT
%pip uninstall -q -y diffusers gradio gradio-client

# Gunakan versi yang kompatibel
%pip install -q "transformers==4.45.2" "huggingface-hub==0.36.2" "pytorch-metric-learning>=2,<3" "wandb>=0.17" "sentencepiece" "scikit-learn" "pandas"

# Periksa hasil instalasi
import transformers
import huggingface_hub
import wandb
import pytorch_metric_learning
from transformers import AutoTokenizer, AutoModel

print("Transformers:", transformers.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)
print("W&B:", wandb.__version__)
print("Setup library berhasil!")


Torch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4
Fri Oct 23 04:51:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--

In [ ]:
# 4 — Periksa dataset dan kurangi hanya batch inferensi agar muat di GPU Colab
EVAL = REPO / 'evaluation'
DICT = EVAL / 'xl_bel/umls_onto_all_lang_cased_wikimed_only_399931.txt'
TEST_DE = EVAL / 'xl_bel/xlbel_v0.0/de_1k_test_query.txt'
for path in (DICT, TEST_DE):
    if not path.is_file():
        raise FileNotFoundError(f'Dataset belum ada: {path}')
    print(path.relative_to(REPO), 'bytes:', path.stat().st_size)

WRAPPER = REPO / 'src/model_wrapper.py'
src = WRAPPER.read_text(encoding='utf-8')
compile(src, str(WRAPPER), 'exec')  # hentikan lebih awal jika source sebelumnya rusak
# Hanya default batch pada fungsi embed_dense yang diubah (bukan tokenizer/model/pooling)
pattern = r'(def embed_dense\([^\n]*?batch_size=)(\d+)'
match = re.search(pattern, src)
if not match:
    raise RuntimeError('Tanda tangan embed_dense berbeda. Periksa file manual; patch TIDAK diterapkan.')
current_batch = int(match.group(2))
EVAL_BATCH = min(64, current_batch)
if current_batch > EVAL_BATCH:
    src = re.sub(pattern, lambda m: m.group(1) + str(EVAL_BATCH), src, count=1)
    WRAPPER.write_text(src, encoding='utf-8')
print('Batch inferensi default:', current_batch, '→', EVAL_BATCH)
print('Tokenizer tetap sesuai source repository; tidak diubah oleh regex.')

evaluation/xl_bel/umls_onto_all_lang_cased_wikimed_only_399931.txt bytes: 13034345
evaluation/xl_bel/xlbel_v0.0/de_1k_test_query.txt bytes: 26014
Batch inferensi default: 64 → 64
Tokenizer tetap sesuai source repository; tidak diubah oleh regex.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 5 — Pakai checkpoint resmi: prioritaskan hasil unduhan lama, simpan sekali di Drive
from huggingface_hub import snapshot_download
MODEL_ID = 'cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR'
OLD_MODEL = DRIVE_REPO / 'evaluation' / 'sapbert_model'
STORED_MODEL = STORE / 'models' / 'official_xlmr_2020ab'

def model_complete(p):
    return (p / 'config.json').is_file() and (bool(list(p.glob('*.safetensors'))) or bool(list(p.glob('*.bin'))))

if model_complete(OLD_MODEL):
    persistent_model = OLD_MODEL
    print('Menggunakan checkpoint lama di Drive:', persistent_model)
else:
    persistent_model = STORED_MODEL
    if not model_complete(persistent_model):
        print('Download checkpoint resmi ke Drive (sekali saja) ...')
        snapshot_download(repo_id=MODEL_ID, local_dir=str(persistent_model))
if not model_complete(persistent_model):
    raise RuntimeError('Checkpoint tidak lengkap; periksa unduhan model di Drive.')

# Membaca model dari SSD runtime biasanya lebih cepat daripada langsung dari Drive.
LOCAL_MODEL = Path('/content/sapbert_official_model')
if not model_complete(LOCAL_MODEL):
    shutil.copytree(persistent_model, LOCAL_MODEL, dirs_exist_ok=True)
print('Model evaluasi:', LOCAL_MODEL)

Model evaluasi: /content/sapbert_official_model


In [ ]:
# 6 — Login W&B untuk logging metrik evaluasi; tanpa token evaluasi tetap jalan
import wandb
wandb_key = get_secret('WANDB_API_KEY')
if not wandb_key:
    wandb_key = getpass.getpass('W&B API key (Enter untuk melewati W&B): ').strip()
LOG_TO_WANDB = bool(wandb_key)
if LOG_TO_WANDB:
    wandb.login(key=wandb_key)
    os.environ['WANDB_MODE'] = 'online'
    print('W&B login berhasil.')
else:
    print('W&B dilewati; log/hasil tetap disimpan ke Google Drive.')
wandb_key = None
WANDB_PROJECT = 'sapbert'  # sama dengan proyek bawaan train/train.py

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jasonenkristo (jasonenkristo-brawijaya-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login berhasil.


In [ ]:
# 7 — Evaluasi bahasa Jerman (langkah 5.1), simpan log, kirim P@1/P@5 ke W&B
run_name = 'official_xlbel_de_' + time.strftime('%Y%m%d_%H%M%S')
log_dir = STORE / 'results' / run_name
log_dir.mkdir(parents=True, exist_ok=True)
local_log = Path('/content') / (run_name + '.log')
command = [
    'python', 'eval.py',
    '--model_dir', str(LOCAL_MODEL),
    '--dictionary_path', str(DICT),
    '--data_dir', str(TEST_DE),
    '--output_dir', str(log_dir),
    '--use_cuda', '--max_length', '25', '--custom_query_loader', '--agg_mode', 'cls'
]
sha = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() if (REPO / '.git').exists() else 'unknown'
run = None
try:
    if LOG_TO_WANDB:
        run = wandb.init(
            project=WANDB_PROJECT, name=run_name, job_type='evaluation',
            config={'model': MODEL_ID, 'lang': 'de', 'dictionary': 'wikimed_399931',
                    'max_length': 25, 'pooling': 'cls', 'eval_batch': EVAL_BATCH,
                    'git_commit': sha},
        )
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    with local_log.open('w', encoding='utf-8') as fout:
        process = subprocess.Popen(command, cwd=EVAL, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        for line in process.stdout:
            print(line, end='')
            fout.write(line)
        return_code = process.wait()
    shutil.copy2(local_log, log_dir / 'eval.log')
    if return_code != 0:
        raise RuntimeError(f'Evaluasi gagal (exit code {return_code}). Lihat {log_dir / "eval.log"}')
    output = local_log.read_text(encoding='utf-8', errors='replace')
    acc1_matches = re.findall(r'acc@1=([0-9.+eE-]+)', output)
    acc5_matches = re.findall(r'acc@5=([0-9.+eE-]+)', output)
    if not acc1_matches or not acc5_matches:
        raise RuntimeError('Proses selesai tetapi acc@1/acc@5 tidak ditemukan. Periksa eval.log.')
    p1, p5 = float(acc1_matches[-1]) * 100, float(acc5_matches[-1]) * 100
    result = {'run': run_name, 'git_commit': sha, 'model': MODEL_ID,
              'language': 'de', 'precision_at_1_percent': p1,
              'precision_at_5_percent': p5, 'eval_batch': EVAL_BATCH}
    (log_dir / 'result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print(f'\nBERHASIL: P@1={p1:.2f}% | P@5={p5:.2f}%')
    print('Hasil di Google Drive:', log_dir)
    if run is not None:
        run.log({'evaluation/P@1_percent': p1, 'evaluation/P@5_percent': p5})
        run.summary.update(result)
except Exception:
    # Jika gagal, log tetap disimpan walau W&B tidak menghasilkan skor.
    if local_log.is_file() and not (log_dir / 'eval.log').exists():
        shutil.copy2(local_log, log_dir / 'eval.log')
    if run is not None:
        run.finish(exit_code=1)
        run = None
    raise
finally:
    if run is not None:
        run.finish()

/content/reproduce_liu/evaluation/../src/metric_learning.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast()
Namespace(model_dir='/content/sapbert_official_model', dictionary_path='/content/reproduce_liu/evaluation/xl_bel/umls_onto_all_lang_cased_wikimed_only_399931.txt', data_dir='/content/reproduce_liu/evaluation/xl_bel/xlbel_v0.0/de_1k_test_query.txt', use_cuda=True, output_dir='/content/drive/MyDrive/sapbert_liu/results/official_xlbel_de_20261021_035121', filter_composite=False, filter_duplicate=False, save_predictions=False, max_length=25, cometa=False, medmentions=False, custom_query_loader=True, agg_mode='cls')
10/21/2026 03:51:27 AM: [ DictionaryDataset! dictionary_path=/content/reproduce_liu/evaluation/xl_bel/umls_onto_all_lang_cased_wikimed_only_399931.txt ]

100%|██████████| 399931/399931 [00:00<00:00, 890515.61it/s]
[reference dictionary loaded]
[loading custom queries...]
10/21/2026

evaluation/P@1_percent,▁
evaluation/P@5_percent,▁
eval_batch,64
evaluation/P@1_percent,33.1
evaluation/P@5_percent,37.1
git_commit,bbfc86ce279f1a841f3d...
language,de
model,cambridgeltl/SapBERT...
precision_at_1_percent,33.1
precision_at_5_percent,37.1
run,official_xlbel_de_20...


## Berikutnya: training SAP, bukan sekarang

Jika evaluasi resmi sudah menampilkan `acc@1` dan `acc@5`, barulah siapkan **UMLS 2020AA berlisensi**, pasangan sinonim multilingual, dan konfigurasi training. `train/train.py` resmi menggunakan `wandb.init(project="sapbert")` dan `wandb.log({"Loss": ...})` di setiap batch; proses training dapat ditampilkan online jika sudah login W&B. Untuk satu GPU T4, ukuran batch training 512 dalam panduan penelitian tidak realistis; gunakan eksperimen **adaptasi satu GPU** terpisah dan dokumentasikan perbedaannya dari paper.

**Sesi berikutnya:** buka notebook ini, jalankan sel 1–7 lagi. Dengan `SOURCE_MODE='github'`, repo akan di-clone ke `/content` jika VM baru atau di-`git pull --ff-only` jika repo masih ada. Model di Drive tidak perlu diunduh ulang. Drive berisi data/hasil, bukan lokasi komputasi utama.

**Kalau GitHub privat belum bisa di-clone:** ubah `SOURCE_MODE='drive'` di sel 1, restart runtime, jalankan ulang. Ini memakai salinan code Drive **tanpa** pembaruan otomatis dari GitHub. Jangan anggap ini sudah `git pull`.

**Penting:** W&B bukan penyimpanan utama checkpoint. Simpan checkpoint dan konfigurasi model di `MyDrive/sapbert_liu/runs/` serta commit hash di log. Jangan commit PAT, W&B key, atau data UMLS berlisensi ke GitHub publik.


In [ ]:

from pathlib import Path

BASE = Path("/content/drive/MyDrive/sapbert_liu")

files = {
    "Data UMLS 2020AA": BASE / "data/2020AA/MRCONSO.RRF",
    "Pasangan sinonim": BASE / "data/umls2020aa_all_pairs.txt",
    "Checkpoint A": BASE / "runs/xlmr_sap_all/config.json",
    "Checkpoint B": BASE / "runs/xlmr_sap_en_zh_muse/config.json",
}

for name, path in files.items():
    print(("ADA   " if path.exists() else "BELUM "), name)

results = BASE / "results"
print("\nFolder hasil evaluasi:")
if results.exists():
    for p in sorted(results.iterdir()):
        print("-", p.name)


BELUM  Data UMLS 2020AA
BELUM  Pasangan sinonim
BELUM  Checkpoint A
BELUM  Checkpoint B

Folder hasil evaluasi:
- official_xlbel_de_20261021_035121


In [ ]:

from google.colab import drive
from getpass import getpass
from pathlib import Path
import subprocess
import zipfile

# Mount Google Drive
drive.mount("/content/drive")

# Lokasi penyimpanan permanen
folder = Path("/content/drive/MyDrive/sapbert_liu/data/2020AA")
folder.mkdir(parents=True, exist_ok=True)

filename = "umls-2020AA-mrconso.zip"
target = folder / filename
partial = folder / (filename + ".part")

# Link resmi NLM
source = (
    "https://download.nlm.nih.gov/umls/kss/"
    "2020AA/umls-2020AA-mrconso.zip"
)

if target.exists() and zipfile.is_zipfile(target):
    print("Dataset sudah ada di Drive. Tidak perlu download ulang.")
else:
    key = getpass("Masukkan UMLS API Key: ").strip()
    if not key:
        raise ValueError("API key tidak boleh kosong.")

    print("Mengunduh UMLS 2020AA...")

    cmd = [
        "curl", "-fL",
        "--retry", "8",
        "--retry-all-errors",
        "--retry-delay", "10",
        "--connect-timeout", "30",
        "-C", "-",
        "-G",
        "--data-urlencode", f"url={source}",
        "--data-urlencode", f"apiKey={key}",
        "-o", str(partial),
        "https://uts-ws.nlm.nih.gov/download"
    ]

    result = subprocess.run(cmd)

    if result.returncode != 0:
        raise RuntimeError(
            "Download belum selesai. Periksa API key/koneksi. "
            "Jika server mendukung resume, jalankan cell lagi."
        )

    if not zipfile.is_zipfile(partial):
        raise RuntimeError(
            "File bukan ZIP valid. Periksa API key dan respons server."
        )

    partial.rename(target)
    print("DOWNLOAD BERHASIL!")

if target.exists() and zipfile.is_zipfile(target):
    print("Lokasi:", target)
    print("Ukuran:", round(target.stat().st_size / 1024**2, 1), "MB")
    with zipfile.ZipFile(target) as z:
        print("Contoh isi ZIP:", z.namelist()[:10])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Masukkan UMLS API Key: ··········
Mengunduh UMLS 2020AA...
DOWNLOAD BERHASIL!
Lokasi: /content/drive/MyDrive/sapbert_liu/data/2020AA/umls-2020AA-mrconso.zip
Ukuran: 406.6 MB
Contoh isi ZIP: ['MRCONSO.RRF']


In [ ]:

from google.colab import drive
from pathlib import Path
import zipfile

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/sapbert_liu/data")
ZIP = BASE / "2020AA/umls-2020AA-mrconso.zip"

assert ZIP.exists(), "ZIP tidak ditemukan!"

with zipfile.ZipFile(ZIP) as z:
    info = z.getinfo("MRCONSO.RRF")
    print("Ukuran asli:", round(info.file_size / 1024**3, 2), "GB")

    with z.open("MRCONSO.RRF") as f:
        for _ in range(3):
            kolom = f.readline().decode("utf-8").split("|")
            print("CUI:", kolom[0],
                  "| Bahasa:", kolom[1],
                  "| Istilah:", kolom[14])

print("Pemeriksaan berhasil!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ukuran asli: 1.84 GB
CUI: C0000005 | Bahasa: ENG | Istilah: (131)I-Macroaggregated Albumin
CUI: C0000005 | Bahasa: ENG | Istilah: (131)I-MAA
CUI: C0000005 | Bahasa: FRE | Istilah: Macroagrégats d'albumine marquée à l'iode 131
Pemeriksaan berhasil!


In [ ]:

import io
import random
import zipfile
from itertools import groupby, combinations
from bisect import bisect_right

OUTPUT = BASE / "umls2020aa_all_pairs.txt"
TEMP = BASE / "umls2020aa_all_pairs.txt.part"
rng = random.Random(33)

def parse_row(line):
    fields = line.rstrip("\r\n").split("|")
    return fields[0], fields[1], fields[14]

def make_pairs(names):
    n = len(names)
    total = n * (n - 1) // 2

    if total <= 50:
        yield from combinations(names, 2)
    else:
        starts = [i * (2*n-i-1)//2 for i in range(n-1)]
        for k in rng.sample(range(total), 50):
            i = bisect_right(starts, k) - 1
            j = i + 1 + k - starts[i]
            yield names[i], names[j]

if OUTPUT.exists() and OUTPUT.stat().st_size > 0:
    print("Dataset sudah ada. Tidak diproses ulang.")
else:
    total_rows = 0
    total_cui = 0
    total_pairs = 0

    with zipfile.ZipFile(ZIP) as z:
        with z.open("MRCONSO.RRF") as raw:
            with io.TextIOWrapper(raw, encoding="utf-8") as f:
                rows = (parse_row(line) for line in f)

                with open(TEMP, "w", encoding="utf-8") as out:
                    for cui, group in groupby(rows, key=lambda x: x[0]):
                        seen = set()
                        names = []

                        for _, lang, name in group:
                            total_rows += 1
                            name = name.strip()

                            if name and "||" not in name and name not in seen:
                                seen.add(name)
                                names.append(name)

                        total_cui += 1

                        for a, b in make_pairs(names):
                            out.write(f"{cui}||{a}||{b}\n")
                            total_pairs += 1

                        if total_cui % 500000 == 0:
                            print(f"{total_cui:,} konsep diproses")

    TEMP.replace(OUTPUT)

    print("\nPREPROCESSING BERHASIL!")
    print("Baris MRCONSO:", f"{total_rows:,}")
    print("Jumlah CUI:", f"{total_cui:,}")
    print("Pasangan sinonim:", f"{total_pairs:,}")
    print("File output:", OUTPUT)
    print("Ukuran GB:", round(OUTPUT.stat().st_size / 1024**3, 2))


500,000 konsep diproses
1,000,000 konsep diproses
1,500,000 konsep diproses
2,000,000 konsep diproses
2,500,000 konsep diproses
3,000,000 konsep diproses
3,500,000 konsep diproses
4,000,000 konsep diproses

PREPROCESSING BERHASIL!
Baris MRCONSO: 15,479,756
Jumlah CUI: 4,281,184
Pasangan sinonim: 21,082,913
File output: /content/drive/MyDrive/sapbert_liu/data/umls2020aa_all_pairs.txt
Ukuran GB: 1.99


In [ ]:

from pathlib import Path
import random

STORE = Path("/content/drive/MyDrive/sapbert_liu")
SOURCE = STORE / "data/umls2020aa_all_pairs.txt"
SAMPLE = STORE / "data/smoke_pairs_4096.txt"

assert SOURCE.exists(), "Dataset asli tidak ditemukan!"

if SAMPLE.exists():
    print("Sampel sudah tersedia.")
else:
    selected = set(
        random.Random(33).sample(range(21_082_913), 4096)
    )

    temp = SAMPLE.with_suffix(".txt.part")
    count = 0

    with SOURCE.open(encoding="utf-8") as src, \
         temp.open("w", encoding="utf-8") as dst:

        for i, line in enumerate(src):
            if i in selected:
                dst.write(line)
                count += 1

    assert count == 4096, f"Hanya menemukan {count} baris"
    temp.replace(SAMPLE)
    print("Sampel berhasil dibuat!")

print("Dataset:", SAMPLE)


Sampel berhasil dibuat!
Dataset: /content/drive/MyDrive/sapbert_liu/data/smoke_pairs_4096.txt


In [ ]:

from huggingface_hub import snapshot_download

MODEL = STORE / "models/xlmr_base"

ready = (
    (MODEL / "config.json").exists()
    and (
        list(MODEL.glob("*.safetensors"))
        or list(MODEL.glob("*.bin"))
    )
)

if ready:
    print("XLM-R Base sudah tersedia di Drive.")
else:
    snapshot_download(
        repo_id="FacebookAI/xlm-roberta-base",
        local_dir=str(MODEL),
        allow_patterns=["*.json", "*.model", "*.safetensors"]
    )
    print("Model berhasil disimpan!")

print("Lokasi:", MODEL)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Model berhasil disimpan!
Lokasi: /content/drive/MyDrive/sapbert_liu/models/xlmr_base


In [ ]:
# 8 — Buat dataset pilot 100.000 pasangan (JALANKAN SEBELUM TRAINING)
from pathlib import Path
import random

STORE = Path("/content/drive/MyDrive/sapbert_liu")
SOURCE = STORE / "data/umls2020aa_all_pairs.txt"
PILOT = STORE / "data/sap_pilot_100k.txt"
TEMP = STORE / "data/sap_pilot_100k.txt.part"

assert SOURCE.is_file(), "Dataset 21 juta pasangan tidak ditemukan. Mount Drive dahulu."

if PILOT.exists():
    with PILOT.open(encoding="utf-8") as f:
        count = sum(1 for _ in f)
    assert count == 100_000, f"File pilot belum lengkap: {count} baris"
    print("Dataset pilot sudah tersedia. Tidak dibuat ulang.")
else:
    rng = random.Random(33)
    selected = set(rng.sample(range(21_082_913), 100_000))
    count = 0
    with SOURCE.open(encoding="utf-8") as src, TEMP.open("w", encoding="utf-8") as dst:
        for i, line in enumerate(src):
            if i in selected:
                dst.write(line)
                count += 1
    assert count == 100_000, f"Jumlah sampel tidak sesuai: {count}"
    TEMP.replace(PILOT)
    print("Dataset pilot berhasil dibuat!")

print("Lokasi:", PILOT)
print("Ukuran (MB):", round(PILOT.stat().st_size / 1024**2, 2))


Dataset pilot sudah tersedia. Tidak dibuat ulang.
Lokasi: /content/drive/MyDrive/sapbert_liu/data/sap_pilot_100k.txt
Ukuran (MB): 9.65


In [ ]:
# 9 — Training pilot SAP 100K dari XLM-R Base (JALANKAN SETELAH CELL DATASET)
from pathlib import Path
import os
import subprocess
import json
import torch
import wandb

STORE = Path("/content/drive/MyDrive/sapbert_liu")
REPO = Path("/content/reproduce_liu")
TRAIN = REPO / "train/train.py"
MODEL = STORE / "models/xlmr_base"
SAMPLE = STORE / "data/sap_pilot_100k.txt"
OUTPUT = STORE / "runs/sapbert_pilot_100k"
DONE = OUTPUT / "pilot_completed.json"

assert torch.cuda.is_available(), "Aktifkan GPU T4 terlebih dahulu!"
assert TRAIN.is_file(), "Repository belum tersedia di /content/reproduce_liu."
assert SAMPLE.is_file(), "Dataset pilot belum ada. Jalankan cell sebelumnya!"
assert (MODEL / "config.json").is_file(), "Config XLM-R belum tersedia di Drive."
assert list(MODEL.glob("*.safetensors")) or list(MODEL.glob("*.bin")), "Bobot XLM-R belum tersedia."

with SAMPLE.open(encoding="utf-8") as f:
    n_samples = sum(1 for _ in f)
assert n_samples == 100_000, f"Jumlah data pilot bukan 100K: {n_samples}"

# Yang dimodifikasi hanya salinan sementara, bukan repository GitHub.
code = TRAIN.read_text(encoding="utf-8")
assert "num_workers=16" in code, "Versi train.py berbeda; periksa num_workers manual."
temporary = Path("/content/train_pilot_single_gpu.py")
temporary.write_text(code.replace("num_workers=16", "num_workers=0"), encoding="utf-8")

# True hanya jika proses selesai sukses DAN model akhir sudah disimpan.
def model_saved(path):
    return (path / "config.json").is_file() and bool(
        list(path.glob("*.safetensors")) or list(path.glob("*.bin"))
    )

if DONE.is_file() and model_saved(OUTPUT):
    print("PILOT 100K SUDAH SELESAI. Training tidak diulang.")
    print("Checkpoint:", OUTPUT)
else:
    if model_saved(OUTPUT) and not DONE.exists():
        raise RuntimeError(
            "Ditemukan bobot model tanpa tanda selesai. Periksa apakah training sebelumnya selesai; "
            "jangan menimpanya tanpa verifikasi."
        )
    wandb.login()
    command = [
        "python", str(temporary),
        "--model_dir", str(MODEL),
        "--train_dir", str(SAMPLE),
        "--output_dir", str(OUTPUT),
        "--use_cuda", "--amp", "--pairwise", "--use_miner",
        "--loss", "ms_loss", "--train_batch_size", "8",
        "--epoch", "1", "--learning_rate", "2e-5",
        "--max_length", "25", "--miner_margin", "0.2",
        "--type_of_triplets", "all", "--agg_mode", "cls",
        "--random_seed", "33", "--checkpoint_step", "100000"
    ]
    env = os.environ.copy()
    env["WANDB_MODE"] = "online"
    env["WANDB_NAME"] = "sapbert_pilot_100k"
    env["TOKENIZERS_PARALLELISM"] = "false"

    subprocess.run(command, cwd=str(REPO / "train"), env=env, check=True)
    if not model_saved(OUTPUT):
        raise RuntimeError("Proses training selesai tetapi checkpoint tidak ditemukan.")
    DONE.write_text(json.dumps({
        "status": "completed", "pairs": 100_000, "epochs": 1,
        "batch_size": 8, "model_base": "xlm-roberta-base",
        "note": "pilot subset; bukan reproduksi penuh Liu"
    }, indent=2), encoding="utf-8")
    print("PILOT 100K SELESAI!")
    print("Checkpoint:", OUTPUT)


PILOT 100K SELESAI!
Checkpoint: /content/drive/MyDrive/sapbert_liu/runs/sapbert_pilot_100k


In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

OUTPUT = Path(
    "/content/drive/MyDrive/sapbert_liu/runs/sapbert_pilot_100k"
)

done = OUTPUT / "pilot_completed.json"

weights = (
    list(OUTPUT.glob("*.safetensors"))
    + list(OUTPUT.glob("*.bin"))
)

checkpoints = sorted(OUTPUT.glob("checkpoint_iter_*"))

print("=== STATUS TRAINING PILOT 100K ===")
print("Folder output:", OUTPUT.exists())
print("Penanda selesai:", done.exists())
print("Config model:", (OUTPUT / "config.json").exists())
print("File bobot:", [(p.name, round(p.stat().st_size / 1024**2, 1)) for p in weights])
print("Checkpoint berkala:", len(checkpoints))

if done.exists() and weights:
    print("\nSELESAI: Pilot sudah tersimpan.")
elif weights:
    print("\nPERLU VERIFIKASI: Bobot ada, tetapi penanda selesai tidak ada.")
elif checkpoints:
    print("\nADA CHECKPOINT: Periksa isinya sebelum memulai ulang.")
else:
    print("\nBELUM ADA CHECKPOINT FINAL.")
    print("Periksa apakah training masih berjalan di Colab.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== STATUS TRAINING PILOT 100K ===
Folder output: True
Penanda selesai: True
Config model: True
File bobot: [('model.safetensors', 1060.7)]
Checkpoint berkala: 0

SELESAI: Pilot sudah tersimpan.


In [1]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

STORE = Path("/content/drive/MyDrive/sapbert_liu")
MODEL_PILOT = STORE / "runs/sapbert_pilot_100k"

weights = (
    list(MODEL_PILOT.glob("*.safetensors"))
    + list(MODEL_PILOT.glob("*.bin"))
)

print("=== HASIL PILOT 100K ===")
print("Config:", (MODEL_PILOT / "config.json").exists())
print("Bobot model:", [p.name for p in weights])
print("Penanda selesai:",
      (MODEL_PILOT / "pilot_completed.json").exists())

if weights and (MODEL_PILOT / "config.json").exists():
    print("MODEL TERSEDIA — siap diperiksa untuk evaluasi")
else:
    print("Checkpoint final belum ditemukan")


Mounted at /content/drive
=== HASIL PILOT 100K ===
Config: True
Bobot model: ['model.safetensors']
Penanda selesai: True
MODEL TERSEDIA — siap diperiksa untuk evaluasi
